In [50]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from backtesting import Strategy, Backtest
from datetime import datetime, timedelta
from scipy.stats import linregress
import warnings
warnings.filterwarnings("ignore")
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [51]:
df = yf.download("QQQ", period="10y", interval="1d")
df.columns = df.columns.get_level_values(0)
df = df.reset_index()
df['Date'] = pd.to_datetime(df['Date'])
del df['Volume']
df

[*********************100%***********************]  1 of 1 completed


Price,Date,Close,High,Low,Open
0,2016-05-16,99.522125,99.885480,98.348206,98.441379
1,2016-05-17,98.273697,99.727117,98.012820,99.438298
2,2016-05-18,98.627708,99.158766,97.863731,98.050064
3,2016-05-19,98.115288,98.487961,97.397897,98.245723
4,2016-05-20,99.196045,99.596668,98.450698,98.506603
...,...,...,...,...,...
2510,2026-05-11,713.289978,714.590027,708.909973,710.359985
2511,2026-05-12,707.239990,710.179993,696.640015,708.219971
2512,2026-05-13,714.710022,716.650024,704.830017,709.960022
2513,2026-05-14,719.789978,722.030029,714.219971,714.619995


In [52]:
df['EMA9'] = df.Close.ewm(span=9).mean()
df['EMA31'] = df.Close.ewm(span=31).mean()

In [53]:
df

Price,Date,Close,High,Low,Open,EMA9,EMA31
0,2016-05-16,99.522125,99.885480,98.348206,98.441379,99.522125,99.522125
1,2016-05-17,98.273697,99.727117,98.012820,99.438298,98.828554,98.877775
2,2016-05-18,98.627708,99.158766,97.863731,98.050064,98.746240,98.788986
3,2016-05-19,98.115288,98.487961,97.397897,98.245723,98.532503,98.603923
4,2016-05-20,99.196045,99.596668,98.450698,98.506603,98.729892,98.738104
...,...,...,...,...,...,...,...
2510,2026-05-11,713.289978,714.590027,708.909973,710.359985,689.401670,652.529570
2511,2026-05-12,707.239990,710.179993,696.640015,708.219971,692.969334,655.948971
2512,2026-05-13,714.710022,716.650024,704.830017,709.960022,697.317471,659.621537
2513,2026-05-14,719.789978,722.030029,714.219971,714.619995,701.811973,663.382065


In [54]:
def signal(data):
    signal = [0] * len(df)
    for i in range(2,len(df)):
        if (data.EMA9.iloc[i] > data.EMA31.iloc[i]) &\
        (data.Close.iloc[i-1] < data.EMA9.iloc[i-1]) &\
        (data.Close.iloc[i] > data.EMA9.iloc[i]):
            signal[i] = 1
        elif (data.EMA9.iloc[i] < data.EMA31.iloc[i]) &\
        (data.Close.iloc[i-1] > data.EMA9.iloc[i-1]) &\
        (data.Close.iloc[i] < data.EMA9.iloc[i]):
            signal[i] = 2
        else:
            signal[i] = 0
        df["signal"] = signal
        
signal(df)


In [55]:
def long_entries(x):
    offset = 0.002
    if x['signal']==1:
        return x['Low'] * (1-offset)
    else:
        return np.nan

df['long_entries'] = df.apply(lambda x: long_entries(x), axis=1)

def short_entries(x):
    offset = 0.002
    if x['signal']==2:
        return x['High'] * (1+offset)
    else:
        return np.nan

df['short_entries'] = df.apply(lambda x: short_entries(x), axis=1)


print(df['signal'].value_counts())
df.shape

signal
0    2322
1     138
2      55
Name: count, dtype: int64


(2515, 10)

In [56]:
df.set_index('Date', inplace=True)

In [57]:
bar = 2200
df1 = df[bar:bar+315].copy()

fig = go.Figure(data=[go.Candlestick(x=df1.index,
                open=df1['Open'],
                high=df1['High'],
                low=df1['Low'],
                close=df1['Close'],
                increasing_line_color = 'rgba(19,156,19,0.8)',
                decreasing_line_color = 'rgba(175,07,49,0.8)',
                name = 'QQQ')])

fig.add_scatter(x=df1.index, y=df1['long_entries'], mode="markers",
                marker=dict(size=7, symbol='arrow-up', color="White"),
                name="Long Entries")

fig.add_scatter(x=df1.index, y=df1['short_entries'], mode="markers",
                marker=dict(size=7, symbol='cross', color="gold"),
                name="Short Entries")

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.EMA9, 
                         opacity=0.8, 
                         line=dict(color='white', width=1),
                        name='EMA9'))

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.EMA31, 
                         opacity=0.8, 
                         line=dict(color='gold', width=1),
                        name='EMA31'))

fig.update_layout(autosize=False, width=1100, height=700, 
                  xaxis_rangeslider_visible=False, 
                  template="plotly_dark")

fig.update_yaxes(gridcolor="#171717") 
fig.update_xaxes(gridcolor="#171717")

fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]),
    ])

fig.show()


In [58]:
def SIGNAL():
    return df.signal

class MyStrat(Strategy):
    
    def init(self):
        super().init()
        self.signal = self.I(SIGNAL)
    
    def next(self):
        super().next()
        
        price = self.data.Close[-1]
        
        if self.signal==1: 
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99, tp=1.05*price, sl=0.97*price)
        
        elif self.signal==2:
            if self.position.is_long or not self.position:
                self.position.close()
                self.sell(size=0.99, sl=1.01*price, tp=0.935*price)
                         
bt = Backtest(df, MyStrat, cash=100_000, margin=1, exclusive_orders=True, commission=0.0005)
stats = bt.run()
stats

Start                     2016-05-16 00:00:00
End                       2026-05-15 00:00:00
Duration                   3651 days 00:00:00
Exposure Time [%]                     44.0159
Equity Final [$]                 389463.12158
Equity Peak [$]                  389735.93771
Commissions [$]                    27696.1627
Return [%]                          289.46312
Buy & Hold Return [%]               615.53938
Return (Ann.) [%]                    14.59457
Volatility (Ann.) [%]                12.80058
CAGR [%]                              9.83867
Sharpe Ratio                          1.14015
Sortino Ratio                         2.06371
Calmar Ratio                          0.86265
Alpha [%]                           230.43536
Beta                                   0.0959
Max. Drawdown [%]                   -16.91832
Avg. Drawdown [%]                    -2.34958
Max. Drawdown Duration      337 days 00:00:00
Avg. Drawdown Duration       37 days 00:00:00
# Trades                          

In [59]:
trades = stats['_trades']
trades['CumulativePnL'] = trades['PnL'].cumsum()

fig_trades = go.Figure()

fig_trades.add_trace(go.Scatter(x=trades['EntryTime'], 
                                      y=trades['CumulativePnL'], 
                                      mode='lines', 
                                      name='Cumulative PnL', 
                                      line=dict(color='#00df9a')))

fig_trades.update_layout(title='MA Strategy PnL',
                         template="plotly_dark",
                         autosize=False,
                         width=1100,
                         height=700,
                        )

fig_trades.update_yaxes(gridcolor="#171717")
fig_trades.update_xaxes(gridcolor="#171717")

fig_trades.show()

In [60]:
def randomised_trades(trades):
    cumulative_return = [0]

    for pct in trades['ReturnPct']:
        cumulative_return.append(cumulative_return[-1] + (pct * 100))

    return cumulative_return

simulations = 100
curves = []

for i in range(simulations):
    new_trades = trades.sample(frac=1).reset_index(drop=True)
    equity_curve = randomised_trades(new_trades)
    curves.append(equity_curve)

mc = go.Figure()

for equity_curve in curves:
    mc.add_trace(go.Scatter(y=equity_curve, mode='lines', opacity=0.6, showlegend=False))

mc.update_layout(
    title='MA Strategy Monte Carlo Simulation',
    xaxis_title='Trade Number',
    yaxis_title='Equity',
    template="plotly_dark",
    autosize=False,
    width=1100,
    height=700,
)

mc.update_yaxes(gridcolor="#171717")
mc.update_xaxes(gridcolor="#171717")

mc.show()